# 04 · Replay harness & the RL loop

The replay harness re-runs a task at its base commit with another model and scores the result
(tests + diff overlap). This notebook builds a **real local git repo** and runs a **real replay** —
no stubs. Then it shows the closed RL loop in plan-only mode.

In [1]:
import subprocess, tempfile, os
from pathlib import Path

work = Path(tempfile.mkdtemp())
os.environ["EPISODIC_HOME"] = str(work / ".episodic")
origin = work / "origin"; origin.mkdir()
(origin / "f.py").write_text("def f():\n    return 1\n")
(origin / "test_f.py").write_text("from f import f\n\ndef test_f():\n    assert f() == 1\n")

def git(*a):
    subprocess.run(["git", *a], cwd=origin, check=True, capture_output=True)
git("init", "-q"); git("config", "user.email", "t@t.dev"); git("config", "user.name", "t")
git("add", "-A"); git("commit", "-q", "-m", "base")
sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=origin, capture_output=True, text=True).stdout.strip()
print("origin:", origin, "| base commit:", sha[:10])

origin: /var/folders/rp/r9dzylxd7qv0zd69tjzd_8t80000gn/T/tmpq1513peo/origin | base commit: 865bdeef01


In [2]:
from episodic import replay
from episodic.testing import make_episode

ep = make_episode("ep_replay_demo", intent="edit f.py",
                  remote_url=str(origin), repo_root=str(origin), base_commit=sha, files=("f.py",))
replay.create_replay(ep)
rid = replay.replay_id_for(ep)

# Without --execute the harness only returns a PLAN (no clone, no code runs):
plan = replay.run_replay(rid, "candidate-model")
print("executed:", plan["executed"], "| note:", plan["note"][:60])

executed: False | note: not executed: pass execute=true / --execute to clone the rep


With `execute=True` it clones the repo, runs the recorded test command, and scores. Our toy 'model' is a runner script that edits `f.py`:

In [3]:
runner = work / "runner.py"
runner.write_text("import os, sys\nws = sys.argv[2]\n"
                  "open(os.path.join(ws, 'f.py'), 'a').write('# touched by candidate\\n')\n")
res = replay.run_replay(rid, "candidate", execute=True,
                        runner_cmd=f"python3 {runner} {{model}} {{workspace}} {{prompt_file}}")
print("ran:", res["ran"], "| produced:", res["produced_files"])
print("scores:", res["scores"])

ran: True | produced: ['f.py']
scores: {'tests_pass': 1.0, 'diff_overlap': 1.0, 'total': 1.0}


## The closed loop — `episodic loop`

quality-filter → train → replay-eval on held-out tasks → compare reward vs base → promote.
We run it **plan-only** (no `--execute`) so it trains the candidate and reports the plan without
cloning per-task. Pass `execute=True` to actually run replay-eval and decide promotion.

In [4]:
from episodic import loop, store
from episodic.testing import make_episode

for i in range(8):
    e = make_episode(f"ep_loop_{i}", intent=f"task {i}",
                     remote_url=str(origin), repo_root=str(origin), base_commit=sha,
                     outcome="merged" if i % 2 == 0 else "accepted", files=("f.py",))
    store.save_episode(e)

manifest = loop.run_loop({
    "trainer": "command", "format": "sft", "min_composite": 0.0,
    "holdout_frac": 0.4, "seed": 0, "train_config": {"command": "true"},
    "out": str(work / "loopout"),
})
print("decision:", manifest["decision"], "| executed:", manifest["executed"])
print("train episodes:", len(manifest["train_ids"]), "| holdout:", len(manifest["holdout_ids"]))

decision: dry_run | executed: False
train episodes: 6 | holdout: 2
